In [ ]:
## Packages Import
%matplotlib widget
import copy
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
import numba as nb
@nb.njit(parallel=True, fastmath=True)
def _induced_velocity_numba(p, c1, c2, gamma):
    N = p.shape[0]
    M = c1.shape[0]
    v_total = np.zeros((N, 3), dtype=np.float64)
    eps = 1e-4

    # Parallélisation sur les points d'évaluation
    for i in nb.prange(N):
        px, py, pz = p[i, 0], p[i, 1], p[i, 2]
        
        for j in range(M):
            # Vecteurs r1 et r2
            rx1 = px - c1[j, 0]
            ry1 = py - c1[j, 1]
            rz1 = pz - c1[j, 2]
            
            rx2 = px - c2[j, 0]
            ry2 = py - c2[j, 1]
            rz2 = pz - c2[j, 2]
            
            # Vecteur r0 (c2 - c1)
            r0x = c2[j, 0] - c1[j, 0]
            r0y = c2[j, 1] - c1[j, 1]
            r0z = c2[j, 2] - c1[j, 2]
            
            # Produit vectoriel r1 x r2
            cx = ry1 * rz2 - rz1 * ry2
            cy = rz1 * rx2 - rx1 * rz2
            cz = rx1 * ry2 - ry1 * rx2
            
            cross_norm2 = cx*cx + cy*cy + cz*cz
            r1_norm = np.sqrt(rx1*rx1 + ry1*ry1 + rz1*rz1)
            r2_norm = np.sqrt(rx2*rx2 + ry2*ry2 + rz2*rz2)
            
            # Vérification de la singularité (mask)
            if r1_norm > eps and r2_norm > eps and cross_norm2 > eps**4:
                # Calcul du terme scalaire
                dot_term = r0x * (rx1/r1_norm - rx2/r2_norm) + \
                           r0y * (ry1/r1_norm - ry2/r2_norm) + \
                           r0z * (rz1/r1_norm - rz2/r2_norm)
                
                # Assemblage final
                coeff = gamma[j] / (12.566370614359172) # 4 * pi
                factor = coeff * dot_term / cross_norm2
                
                v_total[i, 0] += cx * factor
                v_total[i, 1] += cy * factor
                v_total[i, 2] += cz * factor
                
    return v_total

In [ ]:
class VLMPanel:
    """
    Vortex Lattice Method panel with ring vortex
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, p, v, w, b):
        """
        Panel is of quadrilateral shape and vertices must be ordered from root
        position on leading-edge proceeding in counterclockwise direction.
        The pannel can be a wing pannel (with an offset ring) or a wake pannel (with the ring being the pannel)
        """
        # Wing geometry
        self.span = b       # wing span [m]

        # nature of the pannel (wing or wake)
        self.nature = w        # 0 for wing, 1 for wake

        # Panel geometry
        self.pnt    = p        # panel vertices
        self.ctr    = None     # control point
        self.normal = None     # normal direction
        self.chord  = None     # chord length [m]
        self.width  = None     # pannel width [m]
        self.area   = None     # panel area   [m**2]
        
        # Vortex parameters
        self.vrt = v            # ring vortex points

        if self.nature == 0 :
            self._get_panel_geom()
      
    #-------------#
    #   Methods   #
    #-------------#
    def _get_panel_geom(self):
        """
        Compute panel geometric parameters such as control point position, normal
        versor, chord length, width and area, starting from panel's vertices.
        """
        p = self.pnt
        v = self.vrt
        # Compute representative panel's geometric parameters
        c_avg = 0.5 * (p[2] - p[1] + p[3] - p[0])  # average chord vector
        w_avg = 0.5 * (p[2] - p[3] + p[1] - p[0])  # average span vector
        
        # Compute position of control point
        cp = (p[0] + p[1] + 3*p[3] + 3*p[2])/8    # three-quarter line

        # Compute normal versor and panel surface
        ai = np.zeros(3)
        for i, pi in enumerate(p):
            qi = p[(i + 1) % len(p)]
            ai += np.cross(pi, qi)
        av = 0.5 * ai
        a = np.linalg.norm(av)
        if a > 0.0:
            n = av / a
        else:
            #Degenerate cells with zero area
            print(f"Degenerate panel {i} with {len(p)} vertices")


        self.ctr    = cp
        self.normal = n
        self.chord  = c_avg
        self.width  = w_avg
        self.area   = a


In [ ]:
## Utilities

def chord_fn(n, ar, b, sym, space, shape, lam=1.8):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise stations
        ar    -> aspect ratio
        b     -> wing span
        sym   -> symmetric or not
        space -> uniform or cosine spanwise spacing
        shape -> wing planform shape, can be "rectangular", "elliptical" or "tapered"
        lam   -> taper ratio, only needed for tapered shape

    Output:
        c     -> chord length distribution along spanwise direction, shape (n,)
    """
    le = (0.5 * b)
    s_min, s_max = 0.0, le
    if space :
        if sym :
            theta = np.linspace(np.pi/2, np.pi, n)
            s = le*(-np.cos(theta))
        else :
            theta = np.linspace(0, np.pi, n)
            s = le*(1-np.cos(theta))/2
    else :
        s = np.linspace(s_min, s_max, n)
    match shape:
        case "rectangular":
            c = b/ar * np.ones(n)
        case "elliptical":
            c = 4*b/(np.pi*ar) * np.sqrt(1 - (s/le)**2)
        case "tapered":
            c = - s*4*(lam - 1)/(ar*(lam + 1)) + 4*le*lam/(ar*(lam + 1))
    return c

def plot3D(surfaces, show_mirror):
    """ 
    Plot all the surface in 3D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
    """
    fig_3d = plt.figure(figsize=(14, 8),constrained_layout=True)
    ax4 = fig_3d.add_subplot(111, projection='3d')
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black', label='Wing')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8)
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6, label='Wake')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6)
            
    ax4.view_init(elev=30, azim=40)
    x_lim = ax4.get_xlim3d()
    y_lim = ax4.get_ylim3d()
    z_lim = ax4.get_zlim3d()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_zlim(z_lim[1],z_lim[0])
    ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
    ax4.set_xlabel("y", fontstyle='italic')
    ax4.set_ylabel("x", fontstyle='italic')
    ax4.set_zlabel("z", fontstyle='italic')
    ax4.xaxis.set_major_locator(MultipleLocator(0.5))
    ax4.yaxis.set_major_locator(MultipleLocator(1))
    ax4.zaxis.set_major_locator(MultipleLocator(0.5))

    ax4.set_title('3D view')
    ax4.legend()


In [ ]:
class VLMSurface:
    """ 
    Vortex Lattice Methode lifting surface
    Contains all the information about a lifting surface and its sheded wake, including its boundary condition
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, origin, plan, boundary, shedding, b, c, alpha, beta, lamb, delta, phi, sym, space, n, m):
        
        # Wing geometry
        self.origin    = origin     # starting point of the wing - will end at the middle if it's a symmetric wing
        self.plan      = plan       # plan in wich the wing is based - 0 for z-x, 1 for y-x
        self.span      = b          # wing span
        self.chord     = c          # chord law distribution
        self.sweep     = lamb       # middle chord sweep angle  [rad]
        self.dihedron  = delta      # dihedral angle            [rad]
        self.twist     = phi        # quarter chord twist angle [rad]
        self.symmetric = sym        # True for a symmetric wing

        # Discretization parameters
        self.N         = n          # number of spanwise pannels before an eventual symmetry
        self.M         = m          # number of chordwise pannels
        self.spacing   = space      # True for a cosine spacing at the tips

        # Simulation parameters
        self.boundary  = boundary   # True create a mirror image to model the free surface on the x-y plan
        self.aoa       = alpha      # geometric angle of attack [rad]   -> axe = -y
        self.drift     = beta       # geometric angle of drift  [rad]   -> axe = -z
        self.tip_shed  = shedding   # enable the tip shedding - both -> both tips ; right -> only right tip ; left -> only left tip ; none -> both disable
                                    # left and right are defined for an horizontal wing, for a vertical one left <-> top and right <-> bottom
        # Mesh
        self.wing        = {"real":None}    # wing - and mirrored wing - corner points
        self.wing_panels = {"real":None}    # wing - and mirrored wing - panels
        self.edges       = {}               # trailing edge, left tip and right tip
        self.wake        = {"real":{"middle":np.array([])},"mirror":{}}    # wake - and mirrored wake - corner points, divided in middle, left and right
        self.wake_panels = {"real":[], "mirror":[]}                        # wake - and mirrored wake - panels, divided in middle, left and right

        # vortex strengths
        self.gamma       = np.array([])
        self.gamma_wake  = {"middle":np.array([]), "right":np.array([]), "left":np.array([])}

        # secondary computations
        self.loads       = None             # 3D loads 
        self.Cl_2d       = None             # lift coef at each section

    #-------------#
    #   Methods   #
    #-------------#

    def _unit_rot(self, v, theta, ax):
        """
        Elemental clockwise rotation of the reference system axes.
        
        Input:
            v     -> vector componets - shape: (N, 3)
            theta -> rotation angle - clockwise rotation: theta>0
            ax    -> rotation axis index

        Output:
            v_rot -> rotated vector components - shape: (N, 3)
        """
        # Assemble rotation matrix
        rot = np.zeros((3, 3))  # rotation matrix, v[0
        # diagonal elements
        rot[ax, ax] = 1.0
        rot[ax-1, ax-1] = np.cos(theta)
        rot[ax-2, ax-2] = np.cos(theta)
        # extra-diagonal elements
        rot[ax-2, ax-1] = -np.sin(theta)
        rot[ax-1, ax-2] = np.sin(theta)

        # Apply rotation
        v_rot = np.sum(rot[None, :, :] * v[:, None, :], axis=2)
        return v_rot

    def _paneling(self,p, r, w, n, b):
        """
        Build a list of panels from a grid

        Input:
            p      -> panels corner points
            r      -> ring corner points - same than panel if it's a wake
            w      -> 0 for wing - 1 for wake
            n      -> number of panels in the spanwise direction            
        Output:
            panels -> list of panels
        """
        b = self.span
        m = round(np.size(p)/(3*(n+1)))-1       # number of panels in chordwise direction
        panels = []    
        for i in range(m):
            for j in range(n):
                pnt = [p[i*(1+n)+j],
                    p[i*(n+1)+j+1],
                    p[(i+1)*(n+1)+j+1],
                    p[(i+1)*(n+1)+j]]    # ordered panel vertices    
                vtx = [r[i*(1+n)+j],
                    r[i*(n+1)+j+1],
                    r[(i+1)*(n+1)+j+1],
                    r[(i+1)*(n+1)+j]]    # ordered ring vertices
                panels.append(VLMPanel(pnt, vtx, w, b))
        return panels

    def _symmetry(self, grid, merge, norm, p, n):
        """
        Symmetry of the geometry with respect to a plane, and merge of the two sides if needed.
        If we merge we assume that the plane pass by the origin and that the grid start from the origin

        Input:
            grid  -> grid points 
            merge -> boolean, if True the two sides are merged together, otherwise they are kept separate
            norm  -> normal vector of the symmetry plane
            p     -> point on the symmetry plane
            n     -> number of panels in spanwise direction, only needed for the merge

        Output:
            grid_sym -> symmetric grid points or the merged grid points if merge is True
        """
        S = np.eye(3) - 2*np.outer(norm, norm)/(norm @ norm)    # symmetry matrix
        grid_off = grid - p                                     # grid with the plane as origin
        grid_sym = grid_off@(S.T)                               # symmetric grid
        grid_sym = grid_sym + p                                 # symmetric grid with the original plane position
        if merge :
            m = round(np.size(grid)/(3*(n+1)))-1       # number of panels in chordwise direction
            new_grid = np.empty((0, 3))
            for i in range(m+1):
                new_grid = np.concatenate([new_grid, grid_sym[i*(n+1):(i+1)*(n+1),:][::-1][:-1], grid[i*(n+1):(i+1)*(n+1),:]])  # merge the two sides
            grid_sym = new_grid
        return grid_sym

    def _build_wing(self):
        """
        Build the wing panels and rings, will create self.wing and self.wing_panels
        """
        m = self.M
        n = self.N
        b     = self.span
        alpha = self.aoa
        beta  = self.drift
        lamb  = self.sweep
        delta = self.dihedron
        c     = self.chord
        org   = self.origin

        # Leading-edge length of the semi-wing
        le = (0.5 * b)
        s_min, s_max = 0.0, le

        # Spanwise wing discretization, taking the symmetry and spacing parameters into account
        if self.spacing :
            if self.symmetric :                                 # if it's symmetric, we cluster only at the wing tip, otherwise we cluster at both the root and the tip
                theta = np.linspace(np.pi/2, np.pi, (n+1))
                s = le*(-np.cos(theta))                         # leading-edge coordinate 
            else :
                theta = np.linspace(0, np.pi, (n+1))
                s = le*(1-np.cos(theta))/2                      # leading-edge coordinate
        else :
            s = np.linspace(s_min, s_max, (n+1))                # leading-edge coordinate
        s = np.tile(s,m+1)                                      # grid points z components

        # Chordwise wing panels discretiration
        i = np.arange(m+1)[:, None]
        j = c[None, :]
        ch = (i/m) * j                                          # chordwise discretization at each spanwise station
        one = np.ones_like(i)
        ch = ch - one*(j/2) + c[0]/2                            # grid points x components, the chord law is centered  
        ch = ch.reshape(-1)
        p_wing = np.column_stack((ch, np.zeros_like(s), s)) 

        # apply the geometrical transformations
        p_swept = np.column_stack((p_wing[:,2]*np.tan(lamb)+p_wing[:,0],p_wing[:,1],p_wing[:,2]))     # apply sweep
        p_yaw   = self._unit_rot(p_swept, -delta, 0)                                                        # apply dihedron
        
        # take care of the symmetry
        if self.symmetric :
            p_yaw  = self._symmetry(p_yaw, merge=True, norm=np.array([0,0,1]), p=np.array([0,0,0]), n=n)
            self.N = 2*n
            n      = self.N           # double n for the rest of the code

        self.wing["real"] = p_yaw
        self._apply_twist()
        p_twist = copy.copy(self.wing["real"])

        # build the rings
        self._build_ring()
        r_yaw = self.wing["real"]

        # change of reference plan
        p_geo = self._change_plan(p_twist)      # fully buildt geometrical wing panel corner points
        r_geo = self._change_plan(r_yaw)        # fully buildt geometrical wing ring corner points

        # translate to the origin
        p_geo = p_geo + org
        r_geo = r_geo + org

        # global rotations
        p_attac = self._unit_rot(p_geo,  -alpha, 1)   # incidence rotation                                  
        p_earth = self._unit_rot(p_attac,  -beta, 2)  # drift rotation
        r_attac = self._unit_rot(r_geo,  -alpha, 1)                                                   
        r_earth = self._unit_rot(r_attac,  -beta, 2)

        self.wing_panels["real"] = self._paneling(p_earth, r_earth, 0, n, b)
        self.wing["real"]        = r_earth
        self.edges["middle"] = r_earth[-(n+1):]                                    # get the trailing edge for the shedding
        self.edges["left"]   = r_earth.reshape(m+1, n+1, 3)[:,0,:].reshape(m+1,3)  # get the left tip
        self.edges["right"]  = r_earth.reshape(m+1, n+1, 3)[:,n,:].reshape(m+1,3)  # get the left tip

        # boundary condition
        if self.boundary:
            p_mir = self._symmetry(p_earth, False, np.array([0,0,1]),np.array([0,0,0]),n)
            r_mir = self._symmetry(r_earth, False, np.array([0,0,1]),np.array([0,0,0]),n)
            self.wing_panels["mirror"] = self._paneling(p_mir, r_mir, 0, n, b)
            self.wing["mirror"]        = r_mir


    def _apply_twist(self):
        """
        Add the twist to the wing
        If phi is a number it apply the twist linearly to have a twist phi at the tip (classic wing construction)
        If phi is an array we assumed it's the value wanted at each spanwise section, we also assume it is well broadcasted
        """
        m    = self.M
        n    = self.N
        phi  = self.twist
        grid = self.wing["real"]
        if isinstance(phi, np.ndarray) :        # phi is already discretized at each station
            phi_j = phi                                                         # spanwise twist distribution
        else :
            phi_j = np.abs(grid[:,2])*phi/grid[-1,2]                                                   # spanwise twist distribution
        p_middle = np.zeros_like(grid[:n+1])
        for j in range(n+1):    
            p_middle[j] = (grid[j] + grid[j+m*(n+1)])/2                        # getting the middle chord point of each section
        p_middle = np.tile(p_middle, (m+1,1))                      # broadcasting, each j section got its middle chord position
        grid_off = grid - p_middle                                 # grid where each j section is in the local frame with the middle chord point as x origin
        idx = np.arange((m+1)*(n+1)).reshape(m+1, n+1)
        for j in range(n+1):    
            grid_off[idx[:,j]] = self._unit_rot(grid_off[idx[:,j]], -phi_j[j], 2)                         # twisted frame for each section
        self.wing["real"] = grid_off + p_middle                                                     # twisted wing corner points

    def _build_ring(self):
        """ 
        Build the ring starting from the panels geometry
        self.wing is now the ring corner points
        """
        r_wing = self.wing["real"]
        m      = self.M
        n      = self.N
        for j in range(n+1):
            for i in range(m):
                r_wing[i*(n+1)+j] = r_wing[i*(n+1)+j] + (r_wing[(i+1)*(n+1)+j]-r_wing[i*(n+1)+j]) * 0.25        # one quarter chord offset
            r_wing[m*(n+1)+j] = r_wing[m*(n+1)+j] + (r_wing[m*(n+1)+j]-r_wing[(m-1)*(n+1)+j])/3
        self.wing["real"] = r_wing
        
    def _change_plan(self, grid):
        """ 
        Change the reference plan (only vertical z-x to horizontal y-x for now)

        Input:
        -> grid: grid that you want to rebase

        Output:
        -> new_grid: the grid based in the good reference plan
        """
        if self.plan==1:
            new_grid = np.column_stack([grid[:,0], -grid[:,2], grid[:,1]])   # z-x plan become y-x plan
            return new_grid
        return grid

    def _update_wake(self,wake):
        """ 
        Update the wake geometry, usefull to avoid this lines in _time_sim

        Input:
            wake -> the new wake - in a list [mid_wake, left_wake, right_wake]
        """
        self.wake["real"]["middle"] = wake[0]
        self.wake["real"]["left"]   = wake[1]
        self.wake["real"]["right"]  = wake[2]
        if self.boundary :                             # updating the mirrored wake
            self.wake["mirror"]["middle"] = self._symmetry(wake[0], False, np.array([0,0,1]), np.array([0,0,0]), 0)
            self.wake["mirror"]["left"]   = self._symmetry(wake[1], False, np.array([0,0,1]), np.array([0,0,0]), 0)
            self.wake["mirror"]["right"]  = self._symmetry(wake[2], False, np.array([0,0,1]), np.array([0,0,0]), 0)



In [ ]:
class VLMSolver:
    """
    Vortex Lattice Method solver
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, surfaces, u_inf, boundary):

        # Flow properties
        self.U        = u_inf      # inflow velocity [m/s]
        self.boundary = boundary   # enable the free surface condition

        # surfaces
        self.surfaces = surfaces   # list of the VLMSurface

        # global parameters
        self.N_tot    = None       # total number of panels 
        self.controls = None       # control points of all the real wings
        self.normals  = None       # normals of all the real wings

        # Linear system
        self.A = None       # influence coefficient matrix
        self.b = None       # right hand side
    
        self.time=0

        self._get_parameters()

    #-------------#
    #   Methods   #
    #-------------#
 
    def _get_parameters(self):
        """ 
        get the global parameters from each surface
        """
        ctrl  = []
        norm  = []
        N_tot = 0
        for surface in self.surfaces :
            N_tot += surface.N * surface.M
            for panel in surface.wing_panels["real"] :
                ctrl.append(panel.ctr)     # we compute the normals and control points vector only once
                norm.append(panel.normal)
        self.N_tot    = N_tot
        self.controls = np.array(ctrl)
        self.normals  = np.array(norm)

    def _induced_velocity(self,p,c1,c2,gamma):
        return _induced_velocity_numba(p,c1,c2,gamma)
    
    def _induced_velocityp (self,p,c1,c2,gamma):
        """
        Return the velocity induced by the vortex segments [c1, c2], of strentgh gamma,
        at the points p

        Input :
        p     -> points where the velocity is computed, shape (N,3)
        c1    -> starting points of the segments, shape (M,3)
        c2    -> ending points of the segments, shape (M,3)
        gamma -> circulation of each segment, shape (M,)

        Output :
        v     -> velocity induced at each point by each segment, shape (N,3)
        """
        t=time.time()
        r1 = p[:, None, :] - c1[None, :, :]    # (N,M,3)
        r2 = p[:, None, :] - c2[None, :, :]    # (N,M,3)
        r0 = c2[None, :, :] - c1[None, :, :]   # (1,M,3)
        r1_norm = np.linalg.norm(r1, axis=2)   # (N,M)
        r2_norm = np.linalg.norm(r2, axis=2)
        cross = np.cross(r1, r2)                # (N,M,3)
        cross_norm2 = np.sum(cross**2, axis=2)  # (N,M)
        # The safe are needed to avoid division by zero, the value of 1.0 is arbitrary since we will set the velocity to zero in these case
        eps = 1e-5
        r2_norm_safe = np.where(r2_norm < eps, 1.0, r2_norm)    
        r1_norm_safe = np.where(r1_norm < eps, 1.0, r1_norm)    
        cross_norm2_safe = np.where(cross_norm2 < eps, 1.0, cross_norm2)
        mask = ((r1_norm > eps) &(r2_norm > eps) &(cross_norm2 > eps))                               # points on or aligned with the segment
        term = np.sum(r0 * (r1 / r1_norm_safe[:, :, None] - r2 / r2_norm_safe[:, :, None]), axis=2)  # (N,M)
        coeff = gamma[None, :] / (4 * np.pi)
        v = coeff[:, :, None] * (cross / cross_norm2_safe[:, :, None]) * term[:, :, None]
        v[~mask] = 0                    # if one point is on or aligned with the segment, its induced velocity is 0
        self.time+=time.time() - t
        return np.sum(v, axis=1)  # (N,3)
    

    def _vectorize (self, grid, gamma, n) :
        """
        Transform the grid corner points into two array of points that define all
        the segments, needed for the _induced_velocity function, and the local circulation
        of each segment

        Input :
        grid  -> 2D grid of the points that defines the segments which induce the velocity
        gamma -> circulation of each vortex ring
        n     -> number of panels in spanwise direction, needed to reshape the grid and gamma in the right way

        Output :
        c1, c2    -> starting and ending points of each segment
        gamma_seg -> local circulation of each segment
        """
        m = round(np.size(grid)/(3*(n+1)))-1                    # number of panels in chordwise direction
        idx = np.arange((m+1)*(n+1)).reshape(m+1, n+1)          # reshape for building the segments points
        c1_j = grid[idx[:, :-1]]                                # spanwise segments
        c2_j = grid[idx[:, 1:]]
        c1_i = grid[idx[:-1, :]]                                # chordwise segments
        c2_i = grid[idx[1:, :]]
        c1 = np.concatenate([c1_j.reshape(-1,3), c1_i.reshape(-1,3)])
        c2 = np.concatenate([c2_j.reshape(-1,3), c2_i.reshape(-1,3)])
        gamma = gamma.reshape(m, n)
        gamma_seg_j =  gamma[1:, :] - gamma[:-1, :]                                     # local circulation at each spanwise segment except the first and last one
        gamma_seg_j_full = np.vstack([gamma[0, :], gamma_seg_j, -gamma[-1, :]])         # all the local circulations of spanwise segments
        gamma_seg_i =  -gamma[:, 1:] + gamma[:, :-1]                                    # local circulation at each chordwise segment except the first and last one        
        gamma_seg_i_full = np.hstack([-gamma[:, [0]], gamma_seg_i, gamma[:, [-1]]])     # all the local circulations of chordwise segments
        gamma_seg = np.concatenate([gamma_seg_j_full.reshape(-1), gamma_seg_i_full.reshape(-1)])    # local circulation of each segment
        return c1, c2, gamma_seg
    
    def _full_vectorize(self, wing):
        """
        Build the segments needed for using _induced_velocity from all the surfaces
        Basically do _vectorize at each surface needed and return the same type of results, directly usable for _induced_velocity

        Input :
            wing      -> True if the wing influence needs to be counted (for the local speed for instance) - False if it doesn't (for the RHS...)

        Output :
            c1, c2    -> starting and ending points of each segment
            gamma_seg -> local circulation of each segment
        """
        surfaces  = self.surfaces
        c1_tot = np.empty((0, 3))
        c2_tot = np.empty((0, 3))
        gamma_tot = np.empty(0)
        for surface in surfaces:
            for k in surface.wing.keys():
                if wing :                               # taking the wing into consideration or not
                    mid_grid  = np.concatenate([surface.wing[k][:surface.M*(surface.N+1)], surface.wake[k]["middle"]])
                    mid_gamma = np.concatenate([surface.gamma, surface.gamma_wake["middle"]])
                else :
                    mid_grid  = surface.wake[k]["middle"]
                    mid_gamma = surface.gamma_wake["middle"]
                c1, c2, gamma_v      = self._vectorize(mid_grid, mid_gamma, surface.N)

                if ( surface.tip_shed == "left" ) or ( surface.tip_shed == "both" ) :
                    c1_l, c2_l, gamma_vl = self._vectorize(surface.wake[k]["left"], surface.gamma_wake["left"], round(len(surface.wake[k]["left"])/(surface.M+1)-1))
                else :
                    c1_l, c2_l, gamma_vl = np.empty((0, 3)), np.empty((0, 3)), np.empty(0)   # in order to let the concatenation work and to be able to modifie the gamma for boundary condition
                if ( surface.tip_shed == "right" ) or ( surface.tip_shed == "both" ) :
                    c1_r, c2_r, gamma_vr = self._vectorize(surface.wake[k]["right"], surface.gamma_wake["right"], round(len(surface.wake[k]["right"])/(surface.M+1)-1))
                else :
                    c1_r, c2_r, gamma_vr = np.empty((0, 3)), np.empty((0, 3)), np.empty(0)
                c1_tot    = np.concatenate([c1_tot, c1, c1_l, c1_r])  
                c2_tot    = np.concatenate([c2_tot, c2, c2_l, c2_r])  
                if k=="real":           # used to put a - before the mirrored gamma if we want a pure symmetric boundary condition (wall), without a minus its a negative image
                    gamma_tot = np.concatenate([gamma_tot, gamma_v, gamma_vl, gamma_vr]) 
                else :
                    gamma_tot = np.concatenate([gamma_tot, gamma_v, gamma_vl, gamma_vr]) 
        return c1_tot, c2_tot, gamma_tot
            
    def _build_A (self):
        """
        Construct the influence coefficients matrix A by computing the velocity induced at each control point by each vortex ring,
        and projecting it on the normal direction of the panel
        """
        n    = self.N_tot
        ctrl = self.controls
        norm = self.normals
        A = np.zeros((n,n))
        j = 0
        for surface in self.surfaces:
            for i in range(surface.N*surface.M):
                v = surface.wing_panels["real"][i].vrt
                c1 = np.array([v[0], v[1],v[2], v[3]]).reshape(-1,3)                
                c2 = np.array([v[1], v[2],v[3], v[0]]).reshape(-1,3)
                v_ring = self._induced_velocity(ctrl, c1, c2, np.array([1,1,1,1]))  # velocity induced at each control point by the vortex ring of panel j (global) / i (local)
                if self.boundary :
                    v = surface.wing_panels["mirror"][i].vrt
                    c1 = np.array([v[0], v[1],v[2], v[3]]).reshape(-1,3)                
                    c2 = np.array([v[1], v[2],v[3], v[0]]).reshape(-1,3)
                    v_ring = v_ring + self._induced_velocity(ctrl, c1, c2, -np.array([1,1,1,1]))    # velocity induced at each control point by the vortex mirror ring of panel j (global) / i (local)
                A[:,j] = np.sum(v_ring * norm, axis=1)                                           # projection of the induced velocity on the normal direction of each panel
                j += 1
        self.A = A


    def _build_b (self):
        """ 
        Construct the RHS b
        """
        n        = self.N_tot
        u_inf    = self.U
        ctrl     = self.controls
        normal   = self.normals
        surfaces = self.surfaces
        u_inf  = np.tile(u_inf, (n,1))   # n would return a 1D array [u_x, u_y, u_z, u_x, u_y, u_z, ...] the tuple parameters is needed to reshape it in a (n, 3) array
        b = np.zeros(n)
        if np.size(surfaces[0].wake["real"]["middle"]) == 0 :
            v = np.tile(np.array([0,0,0]), (n,1))
        else :
            c1, c2, gamma_v = self._full_vectorize(False)
            v = self._induced_velocity(ctrl, c1, c2, gamma_v)                 # velocity induced at each control point by the wake vortex rings
        b = -np.sum((v + u_inf) * normal, axis=1)
        self.b = b


    def _kuttas_loads(self):
        """
        Compute the loads by applying the Kutta-Joukowski theorem to each vortex segment, and summing up all the contributions
        Each surface ends up whith its loads computed
        
        """
        u        = self.U
        surfaces = self.surfaces
        for surface in surfaces:
            m, n        = surface.M, surface.N
            wing_panels = surface.wing_panels
            gamma       = surface.gamma
            u_inf    = np.tile(u, (m*n, 1))          # m*n would return a 1D array [u_x, u_y, u_z, u_x, u_y, u_z, ...] the tuple parameters is needed to reshape it in a (m*n, 3) array
            u_inf_t  = np.tile(u, ((n-1)*m, 1))      # n-1 because at both tip the local circulation is none (we assume steady state for the loads computation)
            circ     = gamma - np.concatenate([np.zeros(n), gamma[:n*(m-1)]])                            # circulation at each bound segment
            circ_t   = gamma.reshape(m, n)[:,1:].reshape(-1) - gamma.reshape(m, n)[:,:-1].reshape(-1)    # circulation at each trailing segment
            width    = []         
            points   = []         
            chords   = []         
            points_t = []        
            for k,p in enumerate(wing_panels["real"]):
                width.append(p.vrt[1]-p.vrt[0])             # width of each bound segment
                points.append((p.vrt[0] + p.vrt[1])/2)      # mid bound segment points
                if k%n != 0 :                                   # tips are not taken into account TODO tip shedding
                    chords.append(p.vrt[0] - p.vrt[3])          # length of each trailing segment
                    points_t.append((p.vrt[0] + p.vrt[3])/2)    # mid trailing segment points
            width    = np.array(width)
            points   = np.array(points)
            chords   = np.array(chords)
            points_t = np.array(points_t)
            c1, c2, gamma_v = self._full_vectorize(True)
            F_trailing = np.cross(u_inf_t + self._induced_velocity(points_t, c1, c2, gamma_v), circ_t[:,None]*chords)          # loads of the trailing segments by Kutta-Joukowski
            F_bound    = np.cross(u_inf + self._induced_velocity(points, c1, c2, gamma_v), circ[:,None]*width)                 # loads of the bound segments by Kutta-Joukowski                                  
            F_tot      = np.sum( np.concatenate([F_bound, F_trailing]), axis = 0 ) 
            S = np.sum(np.array([p.area for p in surface.wing_panels["real"]]).reshape(m,n), axis=0)
            surface.Cl_2d = 2*np.sum(F_bound[:,1].reshape(m, n), axis=0)/S
            surface.loads = F_tot
        
    def _secondary_computation (self):
        """ 
        Compute the loads using the pressure difference between the two sides of each panel, and summing up all the contributions


        """
        u_inf = self.U
        for surface in self.surfaces :
            m, n    = surface.M, surface.N
            u_inf   = self.U
            gamma   = surface.gamma
            gamma_w = surface.gamma_wake
            panels  = surface.wing_panels["real"]
            ctrl    = np.array([p.ctr for p in panels])
            normals = np.array([p.normal for p in panels])
            u_inf   = np.tile(u_inf, (n*m, 1))
            dp      = []
            print("gamma :", gamma)
            d_gam_i = np.concatenate([gamma[n:],gamma_w["middle"][:n]]) - np.concatenate([np.zeros_like(gamma[:n]), gamma[:n*(m-1)]])     # delta circulation at each control point in the chordwise direction   
            # delta circulation at each control point in the spanwise direction
            if surface.tip_shed == "both":  
                n_w = round(len(gamma_w["left"])/m)                            # number of panels in chordwise direction for the tip wake                
                d_gam_j = np.hstack([gamma.reshape(m, n)[:,1:],gamma_w["right"].reshape(m,n_w)[:,0][:,None]]).reshape(-1) - np.hstack([gamma_w["left"].reshape(m,n_w)[:,-1][:,None], gamma.reshape(m, n)[:,:-1]]).reshape(-1)  
            elif surface.tip_shed == "right":
                n_w = round(len(gamma_w["right"])/m)                           # number of panels in chordwise direction for the tip wake
                d_gam_j = np.hstack([gamma.reshape(m, n)[:,1:],gamma_w["right"].reshape(m,n_w)[:,0][:,None]]).reshape(-1) - np.hstack([gamma.reshape(m, n)[:,0][:,None], gamma.reshape(m, n)[:,:-1]]).reshape(-1)
            elif surface.tip_shed == "left": 
                n_w = round(len(gamma_w["left"])/m )                           # number of panels in chordwise direction for the tip wake
                d_gam_j = np.hstack([gamma.reshape(m, n)[:,1:],gamma.reshape(m, n)[:,-1][:,None]]).reshape(-1) - np.hstack([gamma_w["left"].reshape(m,n_w)[:,-1][:,None], gamma.reshape(m, n)[:,:-1]]).reshape(-1)
            else :
                d_gam_j = np.hstack([gamma.reshape(m, n)[:,1:],gamma.reshape(m, n)[:,-1][:,None]]).reshape(-1) - np.hstack([gamma.reshape(m, n)[:,0][:,None], gamma.reshape(m, n)[:,:-1]]).reshape(-1)
            print("dekta gam I : ",d_gam_i)
            print("delta gam J : ",d_gam_j)
            
            taux_i  = np.array([p.chord/(np.linalg.norm(p.chord)**2) for p in panels])          # chordwise unit vector divided by the mean chord length, for each panel
            taux_j  = np.array([p.width/(np.linalg.norm(p.width)**2) for p in panels])          # spanwise unit vector divided by the mean width, for each panel
            S       = np.array([p.area for p in panels])                                        # area of each panel
            c1, c2, gamma_v = self._full_vectorize(False)
            V       = u_inf + self._induced_velocity(ctrl, c1, c2, gamma_v)
            dp      = np.sum(V * (taux_i*d_gam_i[:,None]/2 + taux_j*d_gam_j[:,None]/2), axis=1) 
            dF      = -dp[:,None]*S[:,None]*normals                                  
            F_tot   = np.sum(dF, axis = 0)
            surface.loads = F_tot

    def _time_sim(self, t, dt, distribution):
        """ 
        Do the time stepping simulation with the wake relaxation

        Input :
            t            -> total simulation time
            dt           -> time step length
            distribution -> type of time step distribution
        """
        # Initialization
        u_inf    = self.U
        surfaces = self.surfaces
        boundary = self.boundary
        self._build_A()
        A = self.A
        inv_A = np.linalg.inv(A)
        self._build_b()
        b = self.b
        Gamma_tot = inv_A @ b               # first solve without wake
        n_gamma   = 0                       # give the start of Gamma in Gamma_tot for each surface
        for surface in surfaces:            # wake initialization by getting the edges
            n, m   = surface.N, surface.M  
            surface.gamma = Gamma_tot[n_gamma:n_gamma+n*m] 
            surface.wake["real"]["middle"] = np.copy(surface.edges["middle"])
            surface.wake["real"]["left"]   = np.copy(surface.edges["left"])   
            surface.wake["real"]["right"]  = np.copy(surface.edges["right"])
            n_gamma += n*m
        match distribution :
            case "classic" :
                dta = np.tile(dt, round(t/dt))
            case "cosine" :         # more pannels at the beginning of the simulation, to better capture the starting vortex
                theta = np.linspace(0, np.pi/2, round(t/dt))  
                dta = t*(1-np.cos(theta))
                dta = dta - np.concatenate([np.array([0]), dta[:-1]])
        for s in range(round(t/dt)):
            n_gamma = 0                 # give the start of Gamma in Gamma_tot for each surface
            for surface in surfaces:
                n, m   = surface.N, surface.M 
                wake   = surface.wake["real"]["middle"]
                l_wake = surface.wake["real"]["left"]
                r_wake = surface.wake["real"]["right"]
                # simulate the wing advancement
                wake   = wake + dta[s]*u_inf
                l_wake = l_wake + dta[s]*u_inf
                r_wake = r_wake + dta[s]*u_inf
                # shed one row of ring
                wake   = np.concatenate([surface.edges["middle"], wake])                       # for each part of the wake we add its shedding edge to the newly convect wake
                l_wake = np.hstack([l_wake.reshape(m+1, s+1, 3), surface.edges["left"].reshape(m+1, 1, 3)]).reshape(-1, 3)          # vertical concatenation
                r_wake = np.hstack([surface.edges["right"].reshape(m+1, 1, 3), r_wake.reshape(m+1, s+1, 3)]).reshape(-1, 3)         
                # store the vorteces strentgh of the wake
                Gamma = Gamma_tot[n_gamma:n_gamma+n*m]              # Gamma is the circulation of each panel of this surface
                surface.gamma                = Gamma
                surface.gamma_wake["middle"] = np.concatenate([Gamma[-(n):], surface.gamma_wake["middle"]])         # the newly shed wake panels take the circulation of the previous edge's panels
                surface.gamma_wake["left"]   = np.hstack([surface.gamma_wake["left"].reshape(m, s), Gamma.reshape(m, n)[:,0].reshape(m, 1)]).reshape(-1)            # same with vertical concatenation
                surface.gamma_wake["right"]  = np.hstack([Gamma.reshape(m, n)[:,n-1].reshape(m, 1), surface.gamma_wake["right"].reshape(m, s)]).reshape(-1)
                n_gamma += n*m                                      # updating the position in the total circulation
                # update the wake geometry
                surface._update_wake([wake, l_wake, r_wake])        # updating the wake for the next b computation
            # update the right hand side and gamma copmutation
            self._build_b()
            Gamma_tot = inv_A @ self.b                              # solving the new situation
            n_gamma = 0
            for surface in surfaces:                                # we update the circulations
                surface.gamma = Gamma_tot[n_gamma:n_gamma+surface.N*surface.M]      
                n_gamma += surface.M*surface.N  
            # simulate the wake rollup
            c1, c2, gamma_v = self._full_vectorize(True)            # getting all the segments (wing + wake) that induce velocity
            for surface in surfaces:
                n, m   = surface.N, surface.M 
                wake   = surface.wake["real"]["middle"]
                l_wake = surface.wake["real"]["left"]
                r_wake = surface.wake["real"]["right"]
                wake   = wake + np.concatenate([np.zeros((n+1,3)), dta[s]*self._induced_velocity(wake[n+1:], c1, c2, gamma_v)])    # each corner point except at the edges are convect by the induced velocity
                l_wake = l_wake + np.hstack([
                    dta[s]*self._induced_velocity(l_wake.reshape(m+1, s+2, 3)[:,:s+1].reshape(-1, 3), c1, c2, gamma_v).reshape(m+1, s+1, 3),
                    np.zeros((m+1,1,3))
                    ]).reshape(-1, 3)                               # reshape in 2D grid to not take into account the tip edge
                r_wake = r_wake + np.hstack([
                    np.zeros((m+1,1,3)), 
                    dta[s]*self._induced_velocity(r_wake.reshape(m+1, s+2, 3)[:,1:].reshape(-1, 3), c1, c2, gamma_v).reshape(m+1, s+1, 3),
                    ]).reshape(-1, 3)
                surface._update_wake([wake, l_wake, r_wake])
                # at the last step we build the wake panels, usefull for plotting
                if s == round(t/dt)-1 :
                    span   = surface.span
                    panels = []
                    panels = panels + surface._paneling(surface.wake["real"]["middle"], surface.wake["real"]["middle"], 1, n, span)
                    if ( surface.tip_shed == "left" ) or ( surface.tip_shed == "both" ) :
                        panels = panels + surface._paneling(surface.wake["real"]["left"], surface.wake["real"]["left"], 1, s+1, span)
                    if ( surface.tip_shed == "right" ) or ( surface.tip_shed == "both" ) :
                        panels = panels + surface._paneling(surface.wake["real"]["right"], surface.wake["real"]["right"], 1, s+1, span)
                    surface.wake_panels["real"] = panels
                    if boundary:
                        panels = []
                        panels = panels + surface._paneling(surface.wake["mirror"]["middle"], surface.wake["mirror"]["middle"], 1, n, span)
                        if ( surface.tip_shed == "left" ) or ( surface.tip_shed == "both" ) :
                            panels = panels + surface._paneling(surface.wake["mirror"]["left"], surface.wake["mirror"]["left"], 1, s+1, span)
                        if ( surface.tip_shed == "right" ) or ( surface.tip_shed == "both" ) :
                            panels = panels + surface._paneling(surface.wake["mirror"]["right"], surface.wake["mirror"]["right"], 1, s+1, span)
                        surface.wake_panels["mirror"] = panels



In [ ]:
## Parameters Definition

# Wing geometry
B = 1        # wing span                 [m]
AR = 5       # aspect ration             [-]
ALPHA  = 10  # angle of attack           [deg] - positive definite for counterclockwise rotations about y-axis
BETA   = 0   # drift angle               [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0   # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0    # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0    # twist tip angle           [deg] - negative for washout

SYM   = True           # symmetric wing configuration
SPACE = True          # spacing distribution, False = uniform, True = cos
SHAPE = "rectangular"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
FREE  = True         # free surface enable

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 15      # number of panels in spanwise direction
M = 4      # number of panels in chordwise direction

# Time simulation parameters
T  = 2   # length of the simulation in s
DT = 0.1    # time step lentgh 




In [ ]:
## Test VLMSurface
surface1 = VLMSurface(origin=np.array([-0.3,0,1]),
                     plan=1,
                     boundary=FREE,
                     shedding="both",
                     b=1.2*B,
                     c=chord_fn((N+1), 10, 1.2*B, SYM, SPACE, "elliptical"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(0),
                     delta=np.deg2rad(0),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M)
surface2 = VLMSurface(origin=np.array([0.8,0,1]),
                     plan=1,
                     boundary=FREE,
                     shedding="both",
                     b=0.5*B,
                     c=chord_fn((N+1), 10, 0.5*B, SYM, SPACE, "tapered"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(20),
                     delta=np.deg2rad(0),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M)
surface3 = VLMSurface(origin=np.array([0,0,0]),
                     plan=0,
                     boundary=FREE,
                     shedding="right",
                     b=2*B,
                     c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(LAMBDA),
                     delta=np.deg2rad(DELTA),
                     phi=np.deg2rad(PHI),
                     sym=False,
                     space=SPACE,
                     n=N,
                     m=M)

rectangle = VLMSurface(origin=np.array([0,0,0]),
                     plan=0,
                     boundary=FREE,
                     shedding="right",
                     b=B,
                     c=chord_fn((N+1), AR, B, SYM, SPACE, "rectangular"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(LAMBDA),
                     delta=np.deg2rad(DELTA),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M) 

surface1._build_wing()
surface2._build_wing()
surface3._build_wing()
rectangle._build_wing()

vlm = VLMSolver([surface1, surface2, surface3], np.array([U,0,0]), FREE)

vlm._time_sim(t=T, dt=DT, distribution=TIME)
print("done")

plot3D([surface1, surface2, surface3],True)


In [ ]:
## Cl versus alpha
alpha = np.linspace(0,11,12)
timee = np.linspace(1,8,8)
cl    = []
cl_l  = []
for ti in timee :
    vlm = VLMSolver(b=B,
                c=chord_fn((2+1), AR, B, SYM, SPACE, SHAPE),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                phi=np.deg2rad(PHI),
                sym=SYM,
                space=SPACE,
                u_inf=np.array([U, 0.0, 0.0]),
                n=2,m=2)
    vlm._time_sim(ti, DT, distribution=TIME)
    l = vlm._secondary_computation()[1]
    cl.append(2*l*AR/(B**2))
    l = vlm._kuttas_loads()[1]
    cl_l.append(2*l*AR/(B**2))
print(timee)
print(cl)
print(cl_l)

